In [ ]:
#Import Required Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

In [ ]:
#Load Dataset (CIFAR-10)
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# CIFAR-10 has 10 classes
num_classes = 10

In [ ]:
#Load Pre-trained CNN Model (MobileNetV2)
base_model = keras.applications.MobileNetV2(
    input_shape=(32, 32, 3),
    include_top=False,       # Exclude final classifier
    weights='imagenet'       # Use pretrained ImageNet weights
)

# Freeze the base model layers (don't train them yet)
for layer in base_model.layers:
    layer.trainable = False


In [ ]:
#Add Custom Classifier Layers
model = models.Sequential([
    base_model,                                 # Pretrained feature extractor
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')  # Output layer
])

In [ ]:
#Compile the Model
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [ ]:
#Train Classifier Layers
history = model.fit(x_train, y_train,
                    validation_split=0.1,
                    epochs=5,
                    batch_size=64,
                    verbose=1)


In [ ]:
#Evaluate Model Performance
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n Test Accuracy: {test_acc:.4f}")
print(f" Test Loss: {test_loss:.4f}")


In [ ]:
#Plot Accuracy and Loss
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()